# `test_piecewise_feasibility.py` — visual walkthrough

**Purpose:** document what each test class in `test/test_piecewise_feasibility.py` actually probes, with pictures. Intended as review aid for the PR — **not** merged into master.

The test file stress-tests the claim that `add_piecewise_formulation(sign="<="/">=")` yields the **same feasible region** for `(x, y)` regardless of which method (`lp` / `sos2` / `incremental`) dispatches the formulation, on curves where all three are applicable.

Four test classes:

| class | what it probes | scope |
|---|---|---|
| `TestRotatedObjective` | support-function equivalence — 16 rotation directions | the strong test |
| `TestDomainBoundary` | `x` outside `[x_min, x_max]` is infeasible | LP explicit vs SOS2 implicit |
| `TestPointwiseInfeasibility` | `y` just past `f(x)` is infeasible | targeted sanity check |
| `TestNVariableInequality` | 3-variable: first tuple bounded, rest equality | SOS2 vs incremental only |

Below: one visualization per class.

*Run this notebook from the repository root so that `from test.test_piecewise_feasibility import ...` resolves.*

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from test.test_piecewise_feasibility import (
    CURVES,
    Y_HI,
    Y_LO,
    Curve,
)

## Shared primitive: draw the curve and its feasible region

In [ ]:
def draw_curve_and_region(ax, curve: Curve, *, shade: bool = True) -> None:
    """Plot breakpoints + shade the feasible region (hypograph or epigraph)."""
    xs = np.array(curve.x_pts)
    ys = np.array(curve.y_pts)
    ax.plot(xs, ys, "o-", color="C0", lw=2, label="breakpoints")

    if shade:
        if curve.sign == "<=":
            ax.fill_between(
                xs,
                np.full_like(ys, Y_LO),
                ys,
                alpha=0.15,
                color="C0",
                label=f"feasible: y {curve.sign} f(x)",
            )
        else:
            ax.fill_between(
                xs,
                ys,
                np.full_like(ys, Y_HI),
                alpha=0.15,
                color="C0",
                label=f"feasible: y {curve.sign} f(x)",
            )

    pad_x = 0.15 * (xs.max() - xs.min())
    pad_y = 0.15 * (ys.max() - ys.min()) + 1
    ax.set_xlim(xs.min() - pad_x, xs.max() + pad_x)
    ax.set_ylim(ys.min() - pad_y, ys.max() + pad_y)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.grid(alpha=0.3)

## `TestRotatedObjective` — the strong test

For every direction `(α, β)` on the unit circle, minimize `α·x + β·y` under the PWL. The answer is the **support function** of the feasible region in direction `(α, β)` — and for a convex region, the support function uniquely determines the region. If LP and SOS2/incremental give the same support-function value for 16 directions, their feasible regions are identical.

Each red dot below is the extreme point the solver lands at for one direction. The arrows show the objective-push direction. A failure would manifest as one method's dot landing at a different vertex than the oracle's.

In [ ]:
def panel_rotated_objective(ax, curve: Curve, n_dirs: int = 16) -> None:
    draw_curve_and_region(ax, curve)
    xs, ys = np.array(curve.x_pts), np.array(curve.y_pts)
    cx = 0.5 * (xs.min() + xs.max())
    cy = 0.5 * (ys.min() + ys.max())
    arrow_len = 0.25 * min(xs.max() - xs.min(), (ys.max() - ys.min()) + 5)

    for i in range(n_dirs):
        theta = 2 * np.pi * i / n_dirs
        alpha, beta = np.cos(theta), np.sin(theta)
        ax.annotate(
            "",
            xytext=(cx, cy),
            xy=(cx + arrow_len * alpha, cy + arrow_len * beta),
            arrowprops=dict(arrowstyle="->", color="C3", alpha=0.4, lw=1),
        )
        # Oracle extreme point in this direction
        verts = curve.vertices()
        extreme = min(verts, key=lambda v: alpha * v[0] + beta * v[1])
        ax.plot(*extreme, "o", color="C3", ms=4, alpha=0.7)

    ax.plot([], [], "o", color="C3", alpha=0.7, label=f"{n_dirs} extreme points")
    ax.legend(loc="upper left", fontsize=8)
    ax.set_title(f"{curve.name}  (sign={curve.sign})")


fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
panel_rotated_objective(axes[0], CURVES[0])  # concave-smooth
panel_rotated_objective(axes[1], CURVES[2])  # convex-steep
panel_rotated_objective(axes[2], CURVES[5])  # two-segment
fig.suptitle(
    "TestRotatedObjective — support function sampled at 16 directions", fontsize=12
)
plt.tight_layout();

Notice the **dots cluster at the curve breakpoints** (top edges) and at the **bottom corners** `(x_min, Y_LO)`, `(x_max, Y_LO)`. That's because the feasible region is a polygon: linear objectives always attain their optimum at a vertex.

The 288 pytest items (6 curves × 3 methods × 16 directions) check that all three methods land at the same extreme point for every direction.

## `TestDomainBoundary` — enforce `x ∈ [x_min, x_max]`

LP enforces this with an explicit constraint; SOS2/incremental enforce it implicitly via `sum(λ) = 1`. Two different implementations of the same bound — worth a direct probe.

In [ ]:
def panel_domain_boundary(ax, curve: Curve) -> None:
    draw_curve_and_region(ax, curve)
    xs = np.array(curve.x_pts)
    y_span = ax.get_ylim()
    ax.axvline(xs[0], color="C2", lw=1.5, label=f"x_min={xs[0]}")
    ax.axvline(xs[-1], color="C2", lw=1.5, label=f"x_max={xs[-1]}")
    ax.axvline(xs[0] - 1, color="C3", lw=1.5, ls="--")
    ax.axvline(xs[-1] + 1, color="C3", lw=1.5, ls="--")
    yy = y_span[1] - 0.12 * (y_span[1] - y_span[0])
    ax.text(
        xs[0] - 1, yy, "INFEASIBLE\n(x < x_min)", ha="center", fontsize=8, color="C3"
    )
    ax.text(
        xs[-1] + 1, yy, "INFEASIBLE\n(x > x_max)", ha="center", fontsize=8, color="C3"
    )
    ax.legend(loc="lower center", fontsize=7)
    ax.set_title(f"{curve.name} — domain probe")


fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
panel_domain_boundary(axes[0], CURVES[0])  # concave-smooth
panel_domain_boundary(axes[1], CURVES[1])  # concave-shifted (negative domain)
panel_domain_boundary(axes[2], CURVES[5])  # two-segment
fig.suptitle(
    "TestDomainBoundary — x outside the breakpoint range is infeasible", fontsize=12
)
plt.tight_layout();

## `TestPointwiseInfeasibility` — y just past the curve

Rotated objectives probe *extremes*; this test specifically nudges `y` past `f(x)` by a small margin (`0.01`) and asserts infeasibility. Catches NaN-mask or off-by-one-segment bugs that might accidentally allow slack.

In [ ]:
def panel_pointwise(ax, curve: Curve) -> None:
    draw_curve_and_region(ax, curve)
    xs = np.array(curve.x_pts)
    x_mid = 0.5 * (xs[0] + xs[-1])
    fx = curve.f(x_mid)
    y_bad = fx + 0.01 if curve.sign == "<=" else fx - 0.01
    ax.plot(x_mid, fx, "o", color="C2", ms=9, label=f"on curve: f({x_mid:g})={fx:g}")
    ax.plot(
        x_mid, y_bad, "x", color="C3", ms=14, mew=3, label=f"infeasible: y={y_bad:g}"
    )
    ax.legend(loc="lower right", fontsize=7)
    ax.set_title(f"{curve.name} — nudge past f(x)")


fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
panel_pointwise(axes[0], CURVES[0])  # concave-smooth, sign="<="
panel_pointwise(axes[1], CURVES[2])  # convex-steep, sign=">="
panel_pointwise(axes[2], CURVES[4])  # linear-gte
fig.suptitle(
    "TestPointwiseInfeasibility — y past the curve by 0.01 in the sign direction",
    fontsize=12,
)
plt.tight_layout();

## `TestNVariableInequality` — 3-variable sign split

With three tuples `(fuel, power, heat)` and `sign="<="`:
- `fuel` (the **first** tuple) is **bounded above** by its interpolated value,
- `power` and `heat` (remaining tuples) are **forced to equality** — pinned on the curve.

LP doesn't support N > 2 tuples, so this class compares SOS2 vs incremental only. The 3D plot shows the CHP curve and the 7 test points (one per `power_fix`) that both methods must agree on.

In [ ]:
bp = {
    "power": np.array([0, 30, 60, 100]),
    "fuel": np.array([0, 40, 85, 160]),
    "heat": np.array([0, 25, 55, 95]),
}

fig = plt.figure(figsize=(9, 6.5))
ax = fig.add_subplot(projection="3d")
ax.plot(
    bp["power"], bp["fuel"], bp["heat"], "o-", color="C0", lw=2, label="CHP breakpoints"
)

for p in [0, 15, 30, 45, 60, 80, 100]:
    f = np.interp(p, bp["power"], bp["fuel"])
    h = np.interp(p, bp["power"], bp["heat"])
    ax.plot([p], [f], [h], "o", color="C3", ms=7)
    # drop to base plane
    ax.plot([p, p], [f, 0], [h, h], color="C3", alpha=0.3, lw=0.8)

ax.set_xlabel("power")
ax.set_ylabel("fuel")
ax.set_zlabel("heat")
ax.plot(
    [],
    [],
    "o",
    color="C3",
    label="7 test points — power pinned,\nfuel at upper bound, heat on curve",
)
ax.set_title('TestNVariableInequality — CHP curve (sign="<=")')
ax.legend(loc="upper left", fontsize=8);

## What a failing test would tell you

- **Rotated objective fails**: the methods disagree on the feasible region in some direction. The failure message includes the attained `(x, y)` point — you'd see which extreme point one method landed at that the others didn't.
- **Domain boundary fails**: one method lets `x` escape `[x_min, x_max]`. LP path most likely: the domain-bound constraint was dropped. SOS2 path: the `sum(λ) = 1` constraint was weakened.
- **Pointwise infeasibility fails**: one method accepts a point past the curve. Most often a NaN-mask bug in per-entity formulations, or a wrong segment getting picked.
- **N-variable fails**: the sign split went wrong — either an input leaked into the signed link or the first-tuple convention is misrouting.

All 356 pytest items are currently green at `TOL = 1e-5`.